# Nemotron / GPT-OSS ISL-OSL Benchmark

Use this notebook on Brev to run the benchmark matrix and visualize TTFT, total latency, decode throughput, and target pass/fail.

Default targets:

- TTFT <= 2.0s
- Total latency <= 5.0s
- Decode throughput >= 200 tok/s

Reasoning is off / not requested by default in the included matrix.

## 1. Install Notebook Dependencies

Run this once per Brev environment.

In [ ]:
%pip install -q -r ../requirements-notebook.txt

## 2. Load Helpers

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Safe default: hosted API enabled, local endpoints disabled.
MATRIX_FILE = REPO_ROOT / "precision_matrix.example.csv"

# Use this when all listed local endpoints are already running.
# MATRIX_FILE = REPO_ROOT / "precision_matrix.all-local.example.csv"
RESULTS_DIR = REPO_ROOT / "results"

TTFT_TARGET_S = 2.0
TOTAL_LATENCY_TARGET_S = 5.0
THROUGHPUT_TARGET_TOK_S = 200.0

print("Repo:", REPO_ROOT)
print("Matrix:", MATRIX_FILE)

## 3. Review The Benchmark Matrix

Edit `precision_matrix.example.csv` before running if your endpoints or ports differ. Use `precision_matrix.all-local.example.csv` when every endpoint is live and you want side-by-side results for all profiles.

Typical Brev/local setup:

- Nemotron BF16 endpoint on `http://localhost:8001/v1`
- Nemotron FP8 endpoint on `http://localhost:8002/v1`
- Nemotron NVFP4 endpoint on `http://localhost:8003/v1`
- GPT-OSS 120B endpoint on `http://localhost:8004/v1`

If you only have one endpoint live, keep only that row set to `enabled=true`. Disabled rows are skipped and shown with a `skip_reason` in the summary.

In [ ]:
matrix = pd.read_csv(MATRIX_FILE)
matrix

## 4. Run The Matrix

Set `RUN_BENCHMARK = True` when your endpoints are live. The matrix runner writes detailed CSVs plus a `summary.csv` under `results/precision-matrix-*`.

In [ ]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    cmd = [
        "python3", str(REPO_ROOT / "benchmark_precision_matrix.py"),
        "--matrix", str(MATRIX_FILE),
        "--prompt-dir", str(REPO_ROOT / "of1-testprompts"),
        "--ttft-target-s", str(TTFT_TARGET_S),
        "--total-latency-target-s", str(TOTAL_LATENCY_TARGET_S),
        "--throughput-target-tok-s", str(THROUGHPUT_TARGET_TOK_S),
    ]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=REPO_ROOT)
    print("Exit code:", completed.returncode)
else:
    print("Set RUN_BENCHMARK = True after endpoints are live.")

## 5. Load Latest Results

In [ ]:
def latest_summary_path(results_dir):
    candidates = sorted(results_dir.glob("precision-matrix-*/summary.csv"), key=lambda p: p.stat().st_mtime)
    return candidates[-1] if candidates else None

summary_path = latest_summary_path(RESULTS_DIR)
if summary_path is None:
    print("No matrix summary found yet. Run the benchmark matrix first.")
    summary = pd.DataFrame()
else:
    print("Loading:", summary_path)
    summary = pd.read_csv(summary_path)
    display(summary)

## 6. Visualize Summary

In [ ]:
if summary.empty:
    print("No summary data to plot yet.")
else:
    plot_df = summary.copy()
    numeric_cols = [
        "p50_ttft_s", "p90_ttft_s", "p99_ttft_s",
        "p50_total_latency_s", "p90_total_latency_s", "p99_total_latency_s",
        "p50_decode_tok_s", "p90_decode_tok_s", "p99_decode_tok_s",
    ]
    for col in numeric_cols:
        if col in plot_df.columns:
            plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

    labels = plot_df["label"].astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].bar(labels, plot_df["p90_ttft_s"])
    axes[0].axhline(TTFT_TARGET_S, linestyle="--")
    axes[0].set_title("p90 TTFT (s)")
    axes[0].set_ylabel("seconds")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].bar(labels, plot_df["p90_total_latency_s"])
    axes[1].axhline(TOTAL_LATENCY_TARGET_S, linestyle="--")
    axes[1].set_title("p90 Total Latency (s)")
    axes[1].set_ylabel("seconds")
    axes[1].tick_params(axis="x", rotation=45)

    axes[2].bar(labels, plot_df["p50_decode_tok_s"])
    axes[2].axhline(THROUGHPUT_TARGET_TOK_S, linestyle="--")
    axes[2].set_title("p50 Decode Throughput (tok/s)")
    axes[2].set_ylabel("tok/s")
    axes[2].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

## 7. Pass / Fail View

In [ ]:
if summary.empty:
    print("No summary data to inspect yet.")
else:
    pass_cols = [
        "label", "precision", "skip_reason", "completed", "errors",
        "p90_ttft_s", "p90_ttft_pass",
        "p90_total_latency_s", "p90_total_latency_pass",
        "p50_decode_tok_s", "p50_decode_throughput_pass",
        "meets_p90_targets",
    ]
    available = [col for col in pass_cols if col in summary.columns]
    display(summary[available])

## 8. Inspect Per-Prompt Results

Use this to see which individual prompt caused a miss.

In [ ]:
if summary.empty:
    print("No detail CSVs found yet.")
else:
    detail_frames = []
    for csv_path in summary["output_csv"].dropna():
        path = Path(csv_path)
        if not path.is_absolute():
            path = REPO_ROOT / path
        if path.exists():
            frame = pd.read_csv(path)
            detail_frames.append(frame)

    if not detail_frames:
        print("No detail CSVs found.")
    else:
        details = pd.concat(detail_frames, ignore_index=True)
        cols = [
            "precision_label", "model", "prompt_file", "run_index",
            "ttft_s", "total_latency_s", "output_tokens",
            "decode_tokens_per_s", "meets_targets", "status", "error",
        ]
        display(details[[col for col in cols if col in details.columns]].sort_values(["precision_label", "prompt_file", "run_index"]))

## 9. Side-by-Side By Prompt

This pivots per-prompt results so each model/profile is shown next to the others.

In [ ]:
if 'details' not in globals() or details.empty:
    print("No detail data loaded yet.")
else:
    ok_details = details[details["status"] == "ok"].copy()
    if ok_details.empty:
        print("No successful runs to compare yet.")
    else:
        compare = ok_details.groupby(["precision_label", "model", "prompt_file"], as_index=False).agg(
            ttft_s=("ttft_s", "median"),
            total_latency_s=("total_latency_s", "median"),
            decode_tokens_per_s=("decode_tokens_per_s", "median"),
            output_tokens=("output_tokens", "median"),
            pass_rate=("meets_targets", "mean"),
        )
        for metric in ["ttft_s", "total_latency_s", "decode_tokens_per_s"]:
            print(f"\n{metric}")
            pivot = compare.pivot_table(index="prompt_file", columns="precision_label", values=metric)
            display(pivot.round(3))

## 10. Model/Profile Ranking

A compact ranking view across successful runs.

In [ ]:
if 'details' not in globals() or details.empty:
    print("No detail data loaded yet.")
else:
    ok_details = details[details["status"] == "ok"].copy()
    if ok_details.empty:
        print("No successful runs to rank yet.")
    else:
        ranking = ok_details.groupby(["precision_label", "model"], as_index=False).agg(
            runs=("prompt_file", "count"),
            p50_ttft_s=("ttft_s", "median"),
            p90_ttft_s=("ttft_s", lambda s: s.quantile(0.90)),
            p50_total_latency_s=("total_latency_s", "median"),
            p90_total_latency_s=("total_latency_s", lambda s: s.quantile(0.90)),
            p50_decode_tok_s=("decode_tokens_per_s", "median"),
            pass_rate=("meets_targets", "mean"),
        )
        display(ranking.sort_values(["pass_rate", "p90_total_latency_s"], ascending=[False, True]).round(3))